In [1]:
# build tokenizer.json


import json 

# 1. Load dataset
with open("../data/train-tinyStories-10Mb.txt", "r", encoding="utf-8") as f:
    dataset = f.read()
dataset[:1000]


# 2. Extract special token & clean training text
SPECIAL_TOKEN = "<|endoftext|>"
training_text = dataset.replace(SPECIAL_TOKEN, "")


# Find all unique characters in the clean text
unique_chars = sorted(list(set(training_text)))
unique_chars


# Create base vocabulary (ID -> Char string)
vocab = {i: char for i, char in enumerate(unique_chars)}
char_to_id = {char: i for i, char in vocab.items()}


# Assign a fixed ID for our special token at the end
special_token_id = len(vocab)
vocab[special_token_id] = SPECIAL_TOKEN


# 3. Convert training text to initial token IDs
ids = [char_to_id[c] for c in training_text]


# 4. BPE Loop (Finding merges)
vocab_size = 2**5
num_merges = vocab_size - len(vocab)
merges = {}

def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

current_new_id = len(vocab) # Start assigning IDs after special tokens

for i in range(num_merges):
    stats = get_stats(ids)
    if not stats:
        break
    best = max(stats, key=stats.get)
    
    merges[best] = current_new_id
    vocab[current_new_id] = vocab[best[0]] + vocab[best[1]]
    ids = merge(ids, best, current_new_id)
    current_new_id += 1


# 5. Save everything to a single JSON 
serializable_merges = {f"{k[0]},{k[1]}": v for k, v in merges.items()}
tokenizer_data = {
    "special_tokens": {SPECIAL_TOKEN: special_token_id},
    "vocab": {str(k): v for k, v in vocab.items()},
    "merges": serializable_merges
}

with open(f"{vocab_size}-tokenizer.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer_data, f, ensure_ascii=False, indent=4)

print(f"Training finished! Saved {vocab_size}-tokenizer.json")


Training finished! Saved 32-tokenizer.json


* 1024-tokenizer.json
* 2048-tokenizer.json

In [5]:
# Load tokenizer.json

import json
import re


# 1. Load tokenizer configuration
with open("2048-tokenizer.json", "r", encoding="utf-8") as f:
    config = json.load(f)


# Reconstruct vocabulary and merge maps
vocab = {int(k): v for k, v in config["vocab"].items()}
char_to_id = {v: k for k, v in vocab.items() if len(v) == 1} # Only base chars


# Reconstruct merges map: dict of (int, int) -> int
merges = {}
for k, v in config["merges"].items():
    p1, p2 = map(int, k.split(","))
    merges[(p1, p2)] = v


# Special token info
special_tokens = config["special_tokens"]
special_token_id = list(special_tokens.values())[0]
special_token_str = list(special_tokens.keys())[0]


# 2. Helper functions
def merge_tokens(ids, pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

def encode_chunk(text):
    """Encodes text chunks that do NOT contain special tokens"""
    # Initialize with base character IDs
    ids = [char_to_id[c] for c in text if c in char_to_id]
    
    while len(ids) >= 2:
        # Find adjacent pairs in the current sequence
        pairs = list(zip(ids, ids[1:]))
        # Find the pair that was merged earliest during training
        best_pair = min(pairs, key=lambda p: merges.get(p, float('inf')))
        
        if best_pair not in merges:
            break # No more merge rules apply
            
        ids = merge_tokens(ids, best_pair, merges[best_pair])
    return ids


# 3. Main encode/decode functions
def encode(text):
    """Safely split text by special tokens and encode"""
    # Split text matching <|endoftext|>
    parts = re.split(rf"({re.escape(special_token_str)})", text)
    final_ids = []
    for part in parts:
        if part == special_token_str:
            final_ids.append(special_token_id)
        elif part:
            final_ids.extend(encode_chunk(part))
    return final_ids

def decode(ids):
    """Convert token IDs back to a single string"""
    return "".join(vocab.get(idx, "") for idx in ids)


In [6]:

input_text = "Once upon a time, there was a little girl, her aunt made her a delicious meal<|endoftext|>"

encoded = encode(input_text)
decoded = decode(encoded)

print(f"Encoded IDs: {encoded}")
print(f"Decoded Text: {decoded}")


Encoded IDs: [1044, 1987, 97, 518, 87, 150, 89, 456, 518, 107, 1539, 1634, 1104, 63, 2034, 83]
Decoded Text: Once upon a time, there was a little girl, her aunt made her a delicious meal<|endoftext|>
